# Database Engineering (PostgreSQL Integration)

## Objective
Design and implement a relational PostgreSQL database to store cleaned Google Play Store reviews for Ethiopian banks (CBE, BOA, Dashen).

This notebook:
- Creates database schema
- Connects Python to PostgreSQL
- Inserts cleaned review data
- Verifies data integrity

In [22]:
!pip install psycopg2-binary pandas

In [23]:
import psycopg2
import pandas as pd
import numpy as np

In [24]:
DB_CONFIG = {
    "host": "localhost",
    "database": "bank_reviews",
    "user": "postgres",
    "password": "123456",
    "port": 5432
}

In [29]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print("Connected to PostgreSQL successfully")

Connected to PostgreSQL successfully


In [31]:
schema_query = """
-- Banks table
CREATE TABLE IF NOT EXISTS banks (
    bank_id SERIAL PRIMARY KEY,
    bank_name VARCHAR(100) NOT NULL,
    app_name VARCHAR(150) NOT NULL
);

-- Reviews table
CREATE TABLE IF NOT EXISTS reviews (
    review_id SERIAL PRIMARY KEY,
    bank_id INTEGER REFERENCES banks(bank_id) ON DELETE CASCADE,
    review_text TEXT NOT NULL,
    rating INTEGER CHECK (rating BETWEEN 1 AND 5),
    review_date DATE,
    sentiment_label VARCHAR(20),
    sentiment_score FLOAT,
    identified_theme VARCHAR(100),
    source VARCHAR(50) DEFAULT 'Google Play'
);
"""

cur.execute(schema_query)
conn.commit()

print("Tables created successfully")

Tables created successfully


In [47]:
import pandas as pd

df = pd.read_csv("final_reviews.csv")
df.head()

,review,rating,date,bank,source,clean_text,transformer_sentiment_label,transformer_sentiment_score,vader_compound,tb_polarity
0,"The worst app, also bank am begging for my own...",1,2026-05-16,BOA,Google Play,worst app also bank begging money,NEGATIVE,-0.999812,-0.6249,-0.20
1,was Good 🙏,5,2026-05-16,BOA,Google Play,good,POSITIVE,0.999851,0.4404,0.70
2,cool,5,2026-05-15,BOA,Google Play,cool,POSITIVE,0.999851,0.3182,0.35
3,Its Good,5,2026-05-15,BOA,Google Play,good,POSITIVE,0.999839,0.4404,0.70
4,good,5,2026-05-14,BOA,Google Play,good,POSITIVE,0.999816,0.4404,0.70


In [48]:
df_db = df.copy()

df_db = df_db.rename(columns={
    "transformer_sentiment_label": "sentiment_label",
    "transformer_sentiment_score": "sentiment_score"
})

In [50]:
df_banks = df[['bank']].drop_duplicates()

bank_id_map = {}

for bank in df_banks['bank']:
    app_name = f"{bank} Mobile App"

    cur.execute("""
        INSERT INTO banks (bank_name, app_name)
        VALUES (%s, %s)
        RETURNING bank_id;
    """, (bank, app_name))

    bank_id = cur.fetchone()[0]
    bank_id_map[bank] = bank_id

conn.commit()

print("✅ Banks inserted successfully")
print(bank_id_map)

✅ Banks inserted successfully
{'BOA': 1, 'CBE': 2, 'DASHEN': 3}


In [51]:
insert_query = """
INSERT INTO reviews (
    bank_id,
    review_text,
    rating,
    review_date,
    sentiment_label,
    sentiment_score,
    identified_theme,
    source
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cur.execute(insert_query, (
        bank_id_map[row['bank']],
        row['review'],
        int(row['rating']),
        row['date'],
        row.get('sentiment_label'),
        float(row['sentiment_score']) if pd.notna(row.get('sentiment_score')) else None,
        row.get('identified_theme'),
        "Google Play"
    ))

conn.commit()

print("✅ Reviews inserted successfully")

✅ Reviews inserted successfully


In [52]:
cur.execute("SELECT COUNT(*) FROM reviews;")
print("Total Reviews:", cur.fetchone()[0])

Total Reviews: 1113


In [53]:
cur.execute("""
SELECT b.bank_name, COUNT(r.review_id)
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
""")

cur.fetchall()

[('BOA', 369), ('CBE', 368), ('DASHEN', 376)]

In [54]:
cur.execute("""
SELECT b.bank_name, ROUND(AVG(r.rating),2)
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
""")

cur.fetchall()

[('BOA', Decimal('3.64')),
 ('CBE', Decimal('4.09')),
 ('DASHEN', Decimal('3.97'))]

In [55]:
cur.execute("""
SELECT
    COUNT(*) FILTER (WHERE review_text IS NULL),
    COUNT(*) FILTER (WHERE rating IS NULL),
    COUNT(*) FILTER (WHERE review_date IS NULL)
FROM reviews;
""")

cur.fetchall()

[(0, 0, 0)]

In [56]:
pd.read_sql_query("""
SELECT r.review_text, r.rating, b.bank_name
FROM reviews r
JOIN banks b ON r.bank_id = b.bank_id
LIMIT 10;
""", conn)

C:\Users\Simbo\AppData\Local\Temp\ipykernel_28664\655778829.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql_query("""


,review_text,rating,bank_name
0,"The worst app, also bank am begging for my own...",1,BOA
1,was Good 🙏,5,BOA
2,cool,5,BOA
3,Its Good,5,BOA
4,good,5,BOA
5,it's very good app,5,BOA
6,this app is good but the speed of app is very ...,2,BOA
7,good,5,BOA
8,boa the best,5,BOA
9,bank of absiniya is best bank in ethiopian,5,BOA


In [58]:
cur.close()
conn.close()

print("Connection closed")

Connection closed
